# 02 - Evaluate: prompted base model vs fine-tuned model
Run this in the **same Colab session** as the training notebook (so `adapter/` exists), or upload the project zip and adapter folder first.
Both models receive the same system prompt (which already explains the format), so this is a fair *prompted base vs fine-tuned* comparison.
Every score is computed by code in `metrics.py`.

In [ ]:
%cd /content/interview_bot
import json, random, torch, pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from core import SESSIONS
from metrics import score_turn, aggregate

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained("adapter")
tok.padding_side = "left"
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
base = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb, device_map={"": 0})
model = PeftModel.from_pretrained(base, "adapter")
model.eval()

In [ ]:
rows = [json.loads(l) for l in open("data/test.jsonl")]
items = []
for r in rows:
    for g in r["gold"]:
        prompt = tok.apply_chat_template(r["messages"][:g["idx"]], tokenize=False, add_generation_prompt=True)
        items.append({"mode": r["mode"], "prompt": prompt, "gold": g})
print(len(items), "test turns")
LIMIT = None   # set e.g. 150 for a quick smoke test
if LIMIT: items = items[:LIMIT]

@torch.no_grad()
def generate_all(items, bs=16, max_new=130):
    order = sorted(range(len(items)), key=lambda i: len(items[i]["prompt"]))
    outs = [None] * len(items)
    for s in range(0, len(order), bs):
        idx = order[s:s + bs]
        enc = tok([items[i]["prompt"] for i in idx], return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        gen = model.generate(**enc, max_new_tokens=max_new, do_sample=False, pad_token_id=tok.pad_token_id)
        for j, i in enumerate(idx):
            outs[i] = tok.decode(gen[j, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return outs

In [ ]:
# 1) base model (adapter switched off)
with model.disable_adapter():
    base_out = generate_all(items)
# 2) fine-tuned model
tuned_out = generate_all(items)
print("done")

In [ ]:
def table(outs):
    rec = [(it["mode"], score_turn(o, it["gold"])) for it, o in zip(items, outs)]
    return aggregate(rec)

res_base, res_tuned = table(base_out), table(tuned_out)
frames = []
for name, res in [("base (prompted)", res_base), ("fine-tuned", res_tuned)]:
    for mode, d in res.items():
        for k, v in d.items():
            frames.append({"model": name, "mode": mode, "metric": k, "value": v})
df = pd.DataFrame(frames)
pivot = df.pivot_table(index=["mode", "metric"], columns="model", values="value")
pivot.to_csv("results.csv")
pivot

In [ ]:
import matplotlib.pyplot as plt
key = df[df.metric.isin(["format_ok", "action_match", "content_ok", "single_question", "scorecard_valid"])]
p = key.pivot_table(index="metric", columns="model", values="value", aggfunc="mean")
p.plot(kind="bar", figsize=(8, 4)); plt.ylim(0, 1.05); plt.ylabel("score (higher is better)")
plt.title("Held-out test: base vs fine-tuned (average over modes)"); plt.xticks(rotation=20); plt.tight_layout()
plt.savefig("comparison.png", dpi=150); plt.show()

In [ ]:
# Failure analysis: look at what the fine-tuned model still gets wrong (put these in your report!)
bad = [(it, o) for it, o in zip(items, tuned_out)
       if (lambda m: m.get("action_match") == 0 or m.get("content_ok") == 0 or m.get("no_code_leak") == 0)(score_turn(o, it["gold"]))]
print(len(bad), "of", len(items), "turns have a problem")
for it, o in bad[:8]:
    print("MODE", it["mode"], "| gold action", it["gold"]["action"])
    print("PRED:", o[:300].replace("\n", " | "))
    print("-" * 60)

### Your own messy test (do this yourself - AI-written tests would defeat the purpose)
Run `show_question(mode, seed)` to see which question a seeded session asks, then type a REAL answer in your own style
(typos, half-sentences, wrong facts). Add 20+ entries and report the results separately from the main test.

In [ ]:
def start_session(mode, seed):
    s = SESSIONS[mode](rng=random.Random(seed))
    u0 = s.observe("(ready)"); g0 = s.gold(); s.apply(g0["action"])
    return s, u0, g0

def show_question(mode, seed):
    s, u0, g0 = start_session(mode, seed)
    print(g0["text"])

def run_handwritten(tests):
    out = []
    for t in tests:
        s, u0, g0 = start_session(t["mode"], t["seed"])
        u1 = s.observe(t["answer"]); g1 = s.gold()
        msgs = [{"role": "system", "content": s.system}, {"role": "user", "content": u0},
                {"role": "assistant", "content": g0["full"]}, {"role": "user", "content": u1}]
        prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        enc = tok(prompt, return_tensors="pt", add_special_tokens=False).to("cuda")
        with torch.no_grad():
            gen = model.generate(**enc, max_new_tokens=130, do_sample=False, pad_token_id=tok.pad_token_id)
        pred = tok.decode(gen[0, enc["input_ids"].shape[1]:], skip_special_tokens=True)
        gold = {"mode": t["mode"], "action": g1["action"], "must": g1["must"], "cue_any": g1["cue_any"], "overall": g1["overall"]}
        out.append({"mode": t["mode"], "expected": g1["action"], "pred": pred[:120].replace("\n", " | "), **score_turn(pred, gold)})
    return pd.DataFrame(out)

show_question("TECH", 1)
MY_TESTS = [
    # {"mode": "TECH", "seed": 1, "answer": "write your own messy answer here"},
]
if MY_TESTS:
    display(run_handwritten(MY_TESTS))

### Launch the chat app
Run this cell, then open the public link it prints. (`--dry` runs the rule engine without the model, for UI testing.)

In [ ]:
!python app.py --adapter adapter --share